# Using `PhysicalParams` to Build a SeQUeNCe Simulation

This notebook shows how the `PhysicalParams` class (a Python port of the Caleffi 2017
parameter table) drives a SeQUeNCe simulation end-to-end.

The key idea is that every hardware knob in the SeQUeNCe JSON config can be
**derived** from the paper's physical constants — so you only ever edit one place.

| `PhysicalParams` field/method | SeQUeNCe config key | Meaning |
|---|---|---|
| `p()` | `MemoryArray.efficiency` | Atom-photon entanglement success prob. |
| `t_ch` | `MemoryArray.coherence_time` | Memory decoherence time [s] |
| `attempt_frequency()` | `MemoryArray.frequency` | Max excitation rate [Hz] |
| `fiber_attenuation()` | `qconnections.attenuation` | Photon loss per metre |
| `classical_delay_ps()` | `cconnections.delay` | Propagation delay [ps] |

---
## 1 — Imports

In [ ]:
import sys, os, json, tempfile

# Make the project root importable from the tutorials/ folder
sys.path.insert(0, os.path.abspath(".."))

from utils.constants import PARAMS, PhysicalParams

from sequence.kernel.timeline import Timeline
from sequence.topology.node import QuantumRouter, BSMNode
from sequence.topology.router_net_topo import RouterNetTopo
from sequence.app.request_app import RequestApp

print("Imports OK")

---
## 2 — Inspect the paper's reference parameters

In [ ]:
print(PARAMS)
print()
print(f"p()                  = {PARAMS.p():.4f}        # local entanglement prob (Eq. 1)")
print(f"attempt_frequency()  = {PARAMS.attempt_frequency():.1f} Hz  # max memory excitation rate")
print(f"fiber_attenuation()  = {PARAMS.fiber_attenuation():.2e}    # loss coefficient per metre")
print(f"t_ch                 = {PARAMS.t_ch*1e3:.1f} ms        # coherence time")

---
## 3 — Build the SeQUeNCe memory template from `PARAMS`

In [ ]:
memory_template = PARAMS.to_sequence_memory_template(fidelity=0.95)

print("Memory template derived from PARAMS:")
print(json.dumps(memory_template, indent=2))

---
## 4 — Direct link simulation (2 routers, 50 km)

```
  r1 ──── BSM_r1_r2 ──── r2
       25 km       25 km
```

All quantum and classical channel parameters come straight from `PARAMS`.

In [ ]:
LINK_DISTANCE_M = 50_000   # 50 km
SIM_DURATION_S  = 10

qconn, cconn = PARAMS.to_sequence_link("r1", "r2", LINK_DISTANCE_M, seed=2)

config_2node = {
    "stop_time":   int(SIM_DURATION_S * 1e12),
    "is_parallel": False,
    "templates": {
        "paper_memory": PARAMS.to_sequence_memory_template(fidelity=0.95)
    },
    "nodes": [
        {"name": "r1", "type": "QuantumRouter", "seed": 0,
         "memo_size": 10, "template": "paper_memory"},
        {"name": "r2", "type": "QuantumRouter", "seed": 1,
         "memo_size": 10, "template": "paper_memory"},
    ],
    "qconnections": [qconn],
    "cconnections": [cconn],
}

print("Full config:")
print(json.dumps(config_2node, indent=2))

In [ ]:
def write_config(config: dict) -> str:
    f = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
    json.dump(config, f)
    f.close()
    return f.name


def run_simulation(
    config: dict,
    initiator: str,
    responder: str,
    memo_size: int = 5,
    fidelity: float = 0.8,
    warmup_s: float = 1.0,
) -> dict:
    stop_time = config["stop_time"]
    start_t   = int(warmup_s * 1e12)
    end_t     = stop_time - int(0.5e12)

    cfg_file = write_config(config)
    try:
        topo = RouterNetTopo(cfg_file)
        tl   = topo.tl
        node = tl.get_entity_by_name(initiator)
        app  = RequestApp(node)
        node.set_app(app)
        tl.init()
        app.start(responder, start_t, end_t, memo_size, fidelity)
        tl.run()
        return {
            "entanglements": app.memory_counter,
            "throughput":    app.get_throughput(),
        }
    finally:
        os.unlink(cfg_file)

In [ ]:
result = run_simulation(config_2node, "r1", "r2", memo_size=5, fidelity=0.8)

print(f"Entanglement pairs : {result['entanglements']}")
print(f"Throughput         : {result['throughput']:.4f} pairs/s")

---
## 5 — Multi-hop simulation (3 routers, entanglement swapping)

```
  r1 ──── BSM_12 ──── r2 ──── BSM_23 ──── r3
       20 km              20 km
```

Classical channels must connect **all** node pairs (including r1↔r3) so that
swapping correction messages can be delivered.
The delays are computed automatically from `cf` via `PARAMS.classical_delay_ps()`.

In [ ]:
HOP_DISTANCE_M = 20_000   # 20 km per hop

qconn_12, cconn_12 = PARAMS.to_sequence_link("r1", "r2", HOP_DISTANCE_M, seed=3)
qconn_23, cconn_23 = PARAMS.to_sequence_link("r2", "r3", HOP_DISTANCE_M, seed=4)

# r1 <-> r3 direct classical channel (2 hops of fiber away)
cconn_13 = {
    "node1": "r1",
    "node2": "r3",
    "delay": PARAMS.classical_delay_ps(2 * HOP_DISTANCE_M),
}

config_3node = {
    "stop_time":   int(100e12),   # 100 seconds
    "is_parallel": False,
    "templates": {
        "paper_memory": PARAMS.to_sequence_memory_template(fidelity=0.95)
    },
    "nodes": [
        {"name": "r1", "type": "QuantumRouter", "seed": 0,
         "memo_size": 20, "template": "paper_memory"},
        {"name": "r2", "type": "QuantumRouter", "seed": 1,
         "memo_size": 20, "template": "paper_memory"},
        {"name": "r3", "type": "QuantumRouter", "seed": 2,
         "memo_size": 20, "template": "paper_memory"},
    ],
    "qconnections": [qconn_12, qconn_23],
    "cconnections": [cconn_12, cconn_23, cconn_13],
}

result_3node = run_simulation(
    config_3node, "r1", "r3",
    memo_size=10, fidelity=0.8, warmup_s=1.0,
)

print(f"Entanglement pairs : {result_3node['entanglements']}")
print(f"Throughput         : {result_3node['throughput']:.4f} pairs/s")

---
## 6 — Customising parameters

`PhysicalParams` is a plain dataclass, so you can override individual fields
using `dataclasses.replace` without mutating `PARAMS`.

In [ ]:
from dataclasses import replace

# Scenario: improved coherence time (e.g. cryogenic memory)
cryo_params = replace(PARAMS, t_ch=1.0)   # 1 second instead of 10 ms

qconn_cryo, cconn_cryo = cryo_params.to_sequence_link("r1", "r2", LINK_DISTANCE_M, seed=2)

config_cryo = {
    "stop_time":   int(SIM_DURATION_S * 1e12),
    "is_parallel": False,
    "templates": {
        "cryo_memory": cryo_params.to_sequence_memory_template(fidelity=0.95)
    },
    "nodes": [
        {"name": "r1", "type": "QuantumRouter", "seed": 0,
         "memo_size": 10, "template": "cryo_memory"},
        {"name": "r2", "type": "QuantumRouter", "seed": 1,
         "memo_size": 10, "template": "cryo_memory"},
    ],
    "qconnections": [qconn_cryo],
    "cconnections": [cconn_cryo],
}

result_paper = run_simulation(config_2node, "r1", "r2", memo_size=5, fidelity=0.8)
result_cryo  = run_simulation(config_cryo,  "r1", "r2", memo_size=5, fidelity=0.8)

print(f"Paper params  (t_ch={PARAMS.t_ch*1e3:.0f} ms) : {result_paper['throughput']:.4f} pairs/s")
print(f"Cryo upgrade  (t_ch={cryo_params.t_ch:.0f} s)  : {result_cryo['throughput']:.4f}  pairs/s")

---
## Summary

| Step | What you do | How `PhysicalParams` helps |
|---|---|---|
| Import | `from utils.constants import PARAMS` | Single source of truth for all constants |
| Memory template | `PARAMS.to_sequence_memory_template()` | Maps `p()`, `t_ch`, `attempt_frequency()` |
| Link config | `PARAMS.to_sequence_link(n1, n2, dist)` | Maps `fiber_attenuation()`, `classical_delay_ps()` |
| Customise | `replace(PARAMS, t_ch=1.0)` | Override one field, everything else stays consistent |